#Covered Today:

1. Visualizing Token-by-Token Inference in GPT Models
2. Building Meeting Minutes from Audio with Whisper and Google Colab
3. Building Meeting Minutes with OpenAI Whisper and LLaMA 3.2
4. Wrap-Up: Build a Synthetic Data Generator with Open Source Models

In [1]:
# we start by logging in into huggingface
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [2]:
# now check, if we are connected to T4
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("Tesla T4") >= 1 and gpu_info.find("CUDA") >= 1:
  print("Connected to Tesla T4")
  print(gpu_info)
else:
  print("Not Connected to GPU, please check.")

Not Connected to GPU, please check.


In [3]:
# now, we import pipeline
from transformers import pipeline
import torch

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    dtype = torch.float16,
    device_map="cuda"
  )
audio_filepath = "/content/denver_extract.mp3"

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

In [4]:
result = asr(audio_filepath, return_timestamps=True)
print(result)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

{'text': " kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so I'll let you see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous people's day so thank you thank you so much and thanks for your leadership all right welcome to the denver city council meeting of monday october 9th please rise with the pledge of allegiance by councilman lopez I pledge allegiance to the fla

In [5]:
# now we simply extract this text and save this in a var
open_source_transcript = result["text"]
print(open_source_transcript)

 kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue, so that is the reason that the back of the logo is considered water, so I'll let you see the creation of the logo here. And yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of indigenous people's day so thank you thank you so much and thanks for your leadership all right welcome to the denver city council meeting of monday october 9th please rise with the pledge of allegiance by councilman lopez I pledge allegiance to the flag of the U

In [6]:
# next we use gpt-transcribe for transcription using openai api.
from openai import OpenAI
openai_api = userdata.get("OPENAI_API_KEY")
if openai_api.startswith("sk-proj-"):
  print("OpenAI API Key Valid")
else:
  print("OpenAI API Key Invalid")

OpenAI API Key Valid


In [7]:
openai_client = OpenAI(api_key=openai_api)
# we check if this is working fine or not
result = openai_client.chat.completions.create(model='gpt-4o-mini', messages=[{"role": "user", "content": "tell a joke"}])
print(result.choices[0].message.content)

Why don't scientists trust atoms?

Because they make up everything!


In [8]:
# cool, the API is working fine. We now move on to the audio part. For transcribing, we need to first open the audiofile
audio_file = open(audio_filepath, "rb")
openai_transcription = openai_client.audio.transcriptions.create(model='gpt-transcribe', file=audio_file)
print(openai_transcription.text)

Kind of the confluence of this whole idea of the Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So let you see the creation of the logo here. And then, yeah, so that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism. And you'll hear a little bit more about our Confluence Week is basically highlighting all of these Indigenous events and things that are happening around Denver so that we can kind of bring more people together and kind of share this whole idea of Indigenous Peoples Day. So thank you. Thank you so much, and thanks for your leadership. All right. Welcome to the Denver City Council meeting of Monday, October 9th. Please rise with the Pledge of Allegiance by Councilman Lopez. I pledge allegiance to the flag o

In [9]:
# now, we create a prompt and using this prompt, we shall exract the minutes of the meeting. We will be using both open source and openai models to acheive this.
system_prompt = """
You are an expert executive assistant. Your task is to extract and generate a professional Minutes of Meeting (MoM) from the provided raw transcript.

### STRICT INSTRUCTIONS:
1. Rely ONLY on the facts directly mentioned in the transcript. Do NOT invent, assume, extrapolate, or bring in outside information.
2. If any piece of information (e.g., date, specific attendee, deadline) is not mentioned in the transcript, explicitly state "Not specified" or leave the field blank.
3. Keep the tone professional, objective, and concise.

---

### REQUIRED OUTPUT FORMAT:

**1. MEETING OVERVIEW**
- **Topic/Purpose:** [Extract main topic or "Not specified"]
- **Date & Time:** [Extract if mentioned, else "Not specified"]
- **Attendees:** [List participants mentioned in the transcript]

**2. EXECUTIVE SUMMARY**
- [2-3 concise sentences summarizing the primary purpose and overall outcome]

**3. KEY DISCUSSION POINTS**
- **[Topic 1]:** [Brief factual summary of discussion, arguments, or updates]
- **[Topic 2]:** [Brief factual summary of discussion, arguments, or updates]

**4. DECISIONS MADE**
- [Bulleted list of definitive agreements/decisions confirmed in the text. If none, state "None recorded"]

**5. ACTION ITEMS & NEXT STEPS**
| Action Item / Task | Assignee / Owner | Deadline |
| :--- | :--- | :--- |
| [Specific task] | [Person responsible or "Unassigned"] | [Date/time or "Not specified"] |

**6. OPEN QUESTIONS / UNRESOLVED TOPICS**
- [Items left open, unresolved questions, or blockers. If none, state "None recorded"]

---

### TRANSCRIPT TO PROCESS:
"""
system_prompt += open_source_transcript

In [10]:
# we start by using llama3.1-8b to do this.
model = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# now we also install accelerate, so that we can organize memory automatically
!pip install -U accelerate
# we also install pip install bitsandbytes, since, we will be quantizing this model, even with pipeline, else, it will be too big for our system
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 47.7 MB/s eta 0:00:00


In [11]:
from transformers import BitsAndBytesConfig
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

In [12]:
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "create minutes from the given text"}]
open_source_minutes = pipeline(
    "text-generation",
    model=model,
    dtype=torch.bfloat16,
    device_map="auto",
    model_kwargs={"quantization_config": quant_config}
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [27]:
response = open_source_minutes(messages, max_new_tokens=2000)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [28]:
from IPython.display import display, Markdown
display(Markdown(response[0]['generated_text'][2]['content']))

**MEETING OVERVIEW**
- **Topic/Purpose:** Indigenous Peoples Day proclamation and Confluence Week
- **Date & Time:** Monday, October 9th (time not specified)
- **Attendees:** Council members (Black, Clark, Espinosa, Allen, Gilmore, Cashman, Kanich, Lopez, New, Ortega, Sussman, and the Mr. President)

**EXECUTIVE SUMMARY**
The meeting focused on the proclamation of Indigenous Peoples Day in the City and County of Denver and the celebration of Confluence Week. Council members discussed the significance of Indigenous Peoples Day and the contributions of Indigenous people to the city's history and culture.

**KEY DISCUSSION POINTS**
- **Indigenous Peoples Day Proclamation:** Councilman Lopez read Proclamation No. 1127, Series of 2017, which recognized the cultural and foundational contributions of Indigenous people to the city's history, present, and future. The proclamation also promoted education about these historical and contemporary contributions.
- **Confluence Week:** Councilman Lopez mentioned Confluence Week, a celebration highlighting Indigenous events and people's day, and the importance of bringing people together to share this idea.

**DECISIONS MADE**
- **Adoption of Proclamation:** The Council adopted Proclamation No. 1127, Series of 2017, officially designating October 9, 2017, as Indigenous Peoples' Day.

**ACTION ITEMS & NEXT STEPS**
| Action Item / Task | Assignee / Owner | Deadline |
| :--- | :--- | :--- |
| None recorded |

**OPEN QUESTIONS / UNRESOLVED TOPICS**
- None recorded

In [ ]:
# now, we move to openai api and do this
response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)

In [32]:
display(Markdown(response.choices[0].message.content))

**1. MEETING OVERVIEW**
- **Topic/Purpose:** Observance of Indigenous Peoples Day and related community events
- **Date & Time:** Monday, October 9, 2017
- **Attendees:** Councilman Lopez, Councilman Clark, Council members Black, Clark, Espinosa, Allen, Gilmore, Cashman, Kanich, Lopez, New, Ortega, Sussman, Mr. President

**2. EXECUTIVE SUMMARY**
- The meeting involved a proclamation recognizing October 9, 2017, as Indigenous Peoples Day in Denver, celebrating the contributions of Indigenous peoples to the city. A Halloween parade event was announced by Councilman Clark.

**3. KEY DISCUSSION POINTS**
- **Indigenous Peoples Day Proclamation:** Councilman Lopez read Proclamation No. 1127, which acknowledges the historical significance of Indigenous peoples in Denver and celebrates their contributions to the community.
- **Halloween Parade Announcement:** Councilman Clark invited the community to the first Halloween parade on Broadway in District 7, scheduled for Saturday, October 21, at 6:00 p.m.

**4. DECISIONS MADE**
- Proclamation No. 1127, observing Indigenous Peoples Day on October 9, 2017, was adopted.

**5. ACTION ITEMS & NEXT STEPS**
| Action Item / Task | Assignee / Owner | Deadline |
| :--- | :--- | :--- |
| Promote the Halloween parade on social media | Councilman Clark | October 21, 2017 |

**6. OPEN QUESTIONS / UNRESOLVED TOPICS**
- None recorded